# Project 1: FDA Recall Announcements
**Sally Choi** 

### Project Overview
The U.S. Food and Drug Administration (FDA) maintains the Recalls, Market Withdrawals, and Safety Alerts website to inform the public about products that may pose health or safety risks. It publishes company announcements/press releases describing affected products, the reasons for the action, and steps consumers should take. These announcements can involve issues such as undeclared allergens, contamination, or manufacturing problems. The website covers several product categories, including food, beverages, drugs, cosmetics, etc. For this project, I collected data from its results table on October 20, 2026. (website link: https://www.fda.gov/safety/recalls-market-withdrawals-safety-alerts)

I created an automated web-scraping algorithm to collect seven fields from the results table: Date, Brand Name(s), Product Description, Product Type, Recall Reason Description, Company Name, and Terminated Recall. Each observation represents one listing in the table, which may include multiple products or brands.

I restricted the collection to Food & Beverages and Drugs as they were the majority of products that were recalled. For each product category, the program selected the product type, displayed 100 entries per page, and clicked Next until all results pages had been collected. I did not apply a date filter, visit the separate archive, or extract information from individual
company announcement pages. A 30-second pause was included before changing filters, changing the page size, or moving to the next page, as requested by the wesbite's scrapping policy. 

The results from both categories were combined into one dataset for further analysis.

In [2]:
# Import packages
import time
import pandas as pd

from datetime import datetime
from selenium import webdriver
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### Access and collection method

I used Selenium to interact with the FDA website through Chrome. The program selects the product type from the dropdown menu, changes the table to show 100 entries, and clicks "Next" to collect all additional results.

I included a 30-second pause before changing filters, changing the page size, or clicking Next to space out the automated actions. This pause follows the crawl delay listed in the FDA robots.txt file: "wait 30 seconds before starting a new URL request."

Separately, WebDriverWait allows up to 30 seconds for the updated table rows to appear. This is a maximum waiting time, rather than a fixed pause, so the program continues as soon as the required condition is met.

Access guidance: https://www.fda.gov/robots.txt

In [5]:
# Set up the browser
url = "https://www.fda.gov/safety/recalls-market-withdrawals-safety-alerts"

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 30)

driver.get(url)

row_xpath = '//table[@id="datatable"]/tbody/tr[td[2]/a]'
wait.until(EC.presence_of_element_located(("xpath", row_xpath)))

<selenium.webdriver.remote.webelement.WebElement (session="321104c5d74979ca11e8a08b4f836d09", element="f.915E23E4A1A8B97C988891EAF7468BC7.d.237FE51A5173B6F722B6D2688ED9BEC1.e.84")>

In [6]:
# Specify the info being extracted
# This order matches the tables from the wesbite
columns = [
    "Date",
    "Brand Name(s)",
    "Product Description",
    "Product Type",
    "Recall Reason Description",
    "Company Name",
    "Terminated Recall"]

product_types = ["Food & Beverages", "Drugs"]
records = []

### Wait for table updates

Changing a filter or clicking Next updates the results table. I created a function that waits for the previous row to be removed and for a new row to appear before continuing. This helps prevent the program from extracting the previous page again while the new results are loading.

In [7]:
# Wait until the previous results have been replaced
def wait_for_rows(old_row):
    wait.until(EC.staleness_of(old_row))
    wait.until(EC.presence_of_element_located(("xpath", row_xpath)))

### Display 100 entries

I set the table to display 100 entries per page instead of the 25 entries default setting. Adding this feature reduces the number of pagination clicks needed to collect the results, which in turn reduces the time for data collection. 

The program checks this setting before extracting each page. If 100 is already selected, it continues without changing the setting. The final page may contain fewer than 100 entries.

In [8]:
# Only change the page size if 100 is not already selected
def show_100_entries():
    menu = Select(driver.find_element("xpath", '//select[@name="datatable_length"]'))

    if menu.first_selected_option.get_attribute("value") != "100":
        old_row = driver.find_element("xpath", row_xpath)
        time.sleep(30)
        menu.select_by_visible_text("100")
        wait_for_rows(old_row)

### Extract the table
For each row, the program extracts the 7 listing fields (the fields are the columns defined above).

The function reads each table row and stores its values in a dictionary, which are added to a list that will later become the dataset. I removed extra spaces and line breaks to make the text easier to use.

I used textContent to retrieve the cell text, including columns that may be hidden visually when the browser window is narrow.

I also temporarily stored the brand link URL to help identify duplicate
records. The program collects the link address without opening the
announcement page, but this URL is excluded in the final dataset.

In [9]:
def extract_page():
    rows = driver.find_elements("xpath", row_xpath)

    for row in rows:
        cells = row.find_elements("xpath", "./td")
        record = {}

        for i, column in enumerate(columns):
            text = cells[i].get_attribute("textContent")
            record[column] = " ".join(text.split()) #remove extra space and breaks line from each cell

        record["_url"] = cells[1].find_element("xpath", ".//a").get_attribute("href") #store link w/out opening
        records.append(record)

### Collect every results page

The collection starts with Food & Beverages and then repeats the same process for Drugs. For each category, the program checks the page-size setting(number of entries shown), extracts the current table, and clicks Next. This continues until the Next button is disabled, indicating that the final results page has been reached.

The printed messages show which category and page have been collected.
The program also records the start and finish times of the collection loop.

In [10]:
collection_started = datetime.now()

In [12]:
def collect_product_type(product_type):
    old_row = driver.find_element("xpath", row_xpath)
    menu = Select(driver.find_element(
        "xpath", '//select[@id="edit-field-regulated-product-field"]'))

    time.sleep(30)
    menu.select_by_visible_text(product_type)
    wait_for_rows(old_row)

    page = 1

    while True:
        show_100_entries()
        extract_page()
        print(f"{product_type}: page {page} collected")

        next_button = driver.find_element("xpath", '//*[@id="datatable_next"]')

        if "disabled" in next_button.get_attribute("class"): #stop after collecting final page
            break

        old_row = driver.find_element("xpath", row_xpath)
        time.sleep(30)
        next_button.find_element("xpath", "./a").click()
        wait_for_rows(old_row)
        page += 1

for product_type in product_types:
    collect_product_type(product_type)

collection_finished = datetime.now()

Food & Beverages: page 1 collected
Food & Beverages: page 2 collected
Food & Beverages: page 3 collected
Food & Beverages: page 4 collected
Food & Beverages: page 5 collected
Food & Beverages: page 6 collected
Food & Beverages: page 7 collected
Food & Beverages: page 8 collected
Drugs: page 1 collected
Drugs: page 2 collected


### Create and save the dataset

I combined the records from both searches into a DataFrame. Before
saving, I checked for exact duplicate records using all collected fields,
including the temporary announcement URL, and removed them. 

I then removed the temporary URL column and saved the seven requested
columns as fda_food_drug_recalls.csv. I excluded the DataFrame index because
it is not one of the collected variables.

The collection loop ran on October 20, 2026, from approximately 9:41 p.m.
to 9:48 p.m. The program collected a total of 955 observations, but the final dataset will contain 953 observations since the 2 duplicate rows are removed.

In [ ]:
# Assessing the output of scraping
recalls = pd.DataFrame(records, columns=columns + ["_url"])

# check for duplicates
duplicates_removed = recalls.duplicated().sum()
recalls.loc[recalls.duplicated(keep=False)]

recalls = recalls.drop_duplicates()
recalls = recalls[columns].reset_index(drop=True) #keep only the 7 columns in final dataset

# Check count of rows collected for each product type
category_counts = pd.Series({
    "Food & Beverages": recalls["Product Type"].str.contains(
        "Food & Beverages", regex=False, na=False).sum(),
    "Drugs": recalls["Product Type"].str.contains(
        "Drugs", regex=False, na=False).sum()})

# print(category_counts)

#### Duplicates and Product type counts
There were two duplicates in the data because two announcements appeared in both product type searches: their product type
included both Food & Beverages and Drugs. I removed the repeated copies, keeping one record for each announcement. Both category labels were preserved, so these announcements are included in the counts for both categories; therefore, category counts overlap.

I also compared the number of collected records for each product type with the number of entries displayed on the FDA website. The counts matched for both Food & Beverages and Drugs. However, as mentioned records belonging to both categories(duplicates) were counted in each category but retained only once in the final dataset. 

In [30]:
# Outcome of scraping
print("Collection started:", collection_started)
print("Collection finished:", collection_finished)
print("Duplicate records removed:", duplicates_removed)
print("Observations saved:", len(recalls), "- 777 observations for Food & Beverages, " \
"178 for Drugs Product Type")

# recalls.head()

Collection started: 2026-10-20 21:41:09.705885
Collection finished: 2026-10-20 21:47:55.631136
Duplicate records removed: 2
Observations saved: 953 - 777 observations for Food & Beverages, 178 for Drugs Product Type


## Data quality checks

I checked the dataset for missing values, duplicate rows, invalid date
formats, and product types outside the selected categories.

I replaced empty or whitespace-only cells with missing values directly in
the recalls DataFrame. Blank Terminated Recall values were not interpreted
as active recalls, since these recalls may be ongoing or completed.

I checked the Date column for invalid values and converted it to datetime
format for analysis. I also checked that each Product Type field contained
Food & Beverages or Drugs, since some entries include additional labels.

After completing these checks, I saved the updated dataset to the CSV file.

In [40]:
# Treat empty/whitespace cells as missing
recalls = recalls.replace(r"^\s*$", pd.NA, regex=True)

missing_summary = pd.DataFrame({
    "Observations": len(recalls),
    "Missing values": recalls.isna().sum(),
    "Missing (%)": (recalls.isna().mean() * 100).round(2)})

# missing_summary

missing_count_terminated = recalls["Terminated Recall"].isna().sum()
missing_pct_terminated = recalls["Terminated Recall"].isna().mean() * 100

print(f"There are no missing values in the other columns. "
      f"Terminated Recall has {missing_count_terminated} blank values ({missing_pct_terminated:.2f}%).")


There are no missing values in the other columns. Terminated Recall has 659 blank values (69.15%).


Blank Terminated Recall fields indicate that the announcement is not marked
as terminated in the table. According to the FDA, these recalls may be
ongoing or completed, so I did not classify them as active recalls, but just missing values.

In [41]:
# Review duplicates
duplicates = recalls.duplicated()

print("There are", duplicates.sum(), "duplicate rows")
# checked.loc[duplicates]

There are 0 duplicate rows


In [42]:
# Identify invalid dates before replacing the original column
dates = pd.to_datetime(recalls["Date"], format="%m/%d/%Y", errors="coerce")
invalid_dates = recalls["Date"].notna() & dates.isna()
invalid_dates.sum()

print("All dates in the date column is read as a date") 
recalls.loc[invalid_dates, ["Date", "Brand Name(s)"]]

# Store converted dates in the dataset
recalls["Date"] = dates

print("Earliest date:", recalls["Date"].min()) 
print("Latest date:", recalls["Date"].max())

All dates in the date column is read as a date
Earliest date: 2017-10-19 00:00:00
Latest date: 2026-10-17 00:00:00


In [45]:
# Inspect the categories collected
# print(recalls["Product Type"].value_counts(dropna=False))

# Check that each row includes at least one selected category
expected_type = recalls["Product Type"].str.contains(
    r"Food & Beverages|Drugs", case=False, na=False)

print("Rows without either selected product type:", (~expected_type).sum())
# recalls.loc[~expected_type]

Rows without either selected product type: 0


In [49]:
# Make into final csv
recalls.to_csv("fda_food_drug_recalls.csv", index=False)

### Quality check findings 

The final dataset contains 953 observations. There are no missing values
in the first six columns. Terminated Recall contains 659 blank values, representing 69.15%. These blanks were not classified as active recalls because the FDA states that recalls not marked as terminated may be ongoing or completed.

Two announcements appeared in both product-type searches. I removed the
extra copies while preserving both category labels. No duplicate rows
remained in the final dataset.

All dates were successfully converted to datetime format, ranging from
October 19, 2017, to October 17, 2026. Every observation included Food &
Beverages or Drugs in its Product Type field, and the collected counts for
each category matched the entry counts displayed on the FDA website.
Announcements belonging to both categories were counted in each category
but retained only once in the combined dataset.

After completing the checks, I saved the cleaned dataset as
fda_food_drug_recalls.csv.